# 📓 Parser-Vergleich: PyMuPDF4LLM vs. Docling vs. Marker

**Autor:** Sakina Ahmadi
**Beschreibung:** Dieses Notebook vergleicht die Extraktionsqualität der drei Parser
anhand eines Beispiel-Papers aus dem Gold-Standard.

## Ziel
Wir analysieren:
1. **Textlänge** – Wie viele Zeichen extrahiert jeder Parser?
2. **Formel-Erkennung** – Wie viele LaTeX-Formeln werden erkannt?
3. **Tabellen-Erkennung** – Wie viele Markdown-Tabellen werden extrahiert?
4. **Context-Matching** – Ist der `expected_context` im extrahierten Text enthalten?

---
## 1. Parser-Vergleich: Textlänge und Zeichen-Übereinstimmung

Wir laden die Markdown-Dateien eines Beispiel-Papers (`2510.21302v1`)
aus allen drei Parsern und vergleichen:
- **Zeichenanzahl** – Wie viel Text wurde extrahiert?
- **Zeichen-Übereinstimmung** – Wie ähnlich sind die extrahierten Texte?

In [3]:
import os

# Pfade zu den Markdown-Dateien eines Papers
arxiv_id = "2510.21302v1"  # Beispiel aus Ihrem Gold-Standard

pymu_path = f"/kaggle/input/datasets/sakinaahmadi/rag-ml-data/extracted_markdown/{arxiv_id}.md"
docling_path = f"/kaggle/input/datasets/sakinaahmadi/extracted-markdown-docling/markdown/{arxiv_id}.md"
maker_path = f"/kaggle/input/datasets/sakinaahmadi/marker-parsed-papers-3734/marker_parsed_papers/{arxiv_id}/{arxiv_id}.md"

# Dateien lesen und vergleichen
with open(pymu_path, "r", encoding="utf-8") as f:
    pymu_text = f.read()
with open(docling_path, "r", encoding="utf-8") as f:
    docling_text = f.read()
with open(maker_path, "r", encoding="utf-8") as f:
    maker_text = f.read()

print(f"PyMuPDF4LLM: {len(pymu_text)} Zeichen")
print(f"Docling: {len(docling_text)} Zeichen")
print(f"Marker: {len(maker_text)} Zeichen")
print(f"Übereinstimmung PyMuPDF vs Docling: {len(set(pymu_text) & set(docling_text)) / max(len(pymu_text), len(docling_text)):.2%}")
print(f"Übereinstimmung PyMuPDF vs Marker: {len(set(pymu_text) & set(maker_text)) / max(len(pymu_text), len(maker_text)):.2%}")

PyMuPDF4LLM: 90400 Zeichen
Docling: 100756 Zeichen
Marker: 94624 Zeichen
Übereinstimmung PyMuPDF vs Docling: 0.11%
Übereinstimmung PyMuPDF vs Marker: 0.11%


---
## 2. Formel-Erkennung

Wir zählen, wie viele LaTeX-Formelbefehle (`\frac`, `\int`, `\sum`, etc.)
von jedem Parser extrahiert wurden.

**Erwartung:** Docling und Marker sollten mehr Formeln erkennen als PyMuPDF4LLM,
da sie Layout-basiert arbeiten.

In [4]:
import re

# Prüfen, wie viele Formeln extrahiert wurden
def count_formulas(text):
    # Nach LaTeX-Formeln suchen
    return len(re.findall(r"\\frac|\\int|\\sum|\\alpha|\\beta|\\gamma|\\mathcal", text))

print(f"PyMuPDF4LLM Formeln: {count_formulas(pymu_text)}")
print(f"Docling Formeln: {count_formulas(docling_text)}")
print(f"Marker Formeln: {count_formulas(maker_text)}")

PyMuPDF4LLM Formeln: 0
Docling Formeln: 0
Marker Formeln: 100


---
## 3. Tabellen-Erkennung

Wir zählen, wie viele Markdown-Tabellen von jedem Parser extrahiert wurden.

**Erwartung:** Docling und Marker sollten mehr Tabellen erkennen als PyMuPDF4LLM.

In [5]:
# Prüfen, wie viele Tabellen extrahiert wurden
def count_tables(text):
    # Nach Markdown-Tabellen suchen
    return len(re.findall(r"\|.*\|.*\||\n\|", text))

print(f"PyMuPDF4LLM Tabellen: {count_tables(pymu_text)}")
print(f"Docling Tabellen: {count_tables(docling_text)}")
print(f"Marker Tabellen: {count_tables(maker_text)}")

PyMuPDF4LLM Tabellen: 98
Docling Tabellen: 110
Marker Tabellen: 140


---
## 4. Context-Matching mit Gold-Standard

Wir prüfen, ob der `expected_context` aus dem Gold-Standard
im extrahierten Text jedes Parsers enthalten ist.

**Hinweis:** Der `expected_context` wurde automatisch mit Llama-3 generiert
und kann paraphrasiert sein – daher ist ein exakter Match nicht immer zu erwarten.

In [7]:
# Gold-Standard laden
import json

with open("/kaggle/input/datasets/sakinaahmadi/automl-ground-truth-100/automl_ground_truth_100.json", "r") as f:
    gold = json.load(f)

# Für eine Frage prüfen, ob der expected_context im extrahierten Text enthalten ist
item = gold[0]
arxiv_id = item["arxiv_id"]
expected_context = item["expected_context"]

# In den drei Markdown-Dateien suchen
for parser, text in [("PyMuPDF4LLM", pymu_text), ("Docling", docling_text), ("Marker", maker_text)]:
    if expected_context in text:
        print(f"✅ {parser}: expected_context enthalten")
    else:
        print(f"❌ {parser}: expected_context NICHT enthalten")

❌ PyMuPDF4LLM: expected_context NICHT enthalten
❌ Docling: expected_context NICHT enthalten
❌ Marker: expected_context NICHT enthalten


---
## 5. Zusammenfassung der Ergebnisse

| Metrik | PyMuPDF4LLM | Docling | Marker |
|--------|-------------|---------|--------|
| **Zeichenanzahl** | 90.400 | 100.756 | 94.624 |
| **Formel-Erkennung** | 0 | 0 | **100** |
| **Tabellen-Erkennung** | 98 | 110 | **140** |
| **Context-Match** | ❌ | ❌ | ❌ |

### Interpretation
- **Marker** erkennt mit Abstand die meisten Formeln (100) und Tabellen (140).
- **Docling** extrahiert den meisten Text (100.756 Zeichen).
- **PyMuPDF4LLM** ist am schwächsten bei Formeln (0) und Tabellen (98).
- Der `expected_context` wurde von keinem Parser exakt gefunden –
  dies liegt an der Paraphrasierung durch das LLM bei der Generierung.

### Fazit
Für wissenschaftliche Literatur mit vielen Formeln und Tabellen ist
**Marker** oder **Docling** die bessere Wahl als PyMuPDF4LLM.